**Putusan structured-extractor SFT (Stage 1)** — `unsloth/Qwen3.5-9B`, language-only. Reconstructs the model input from each decision's own section spans and finetunes it to emit the 31-section JSON (see [`RAG/ORCHESTRATION.md`](../RAG/ORCHESTRATION.md)). Designed for an **A100 80GB** (putusan contexts are long).
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

To install Unsloth on your local device, follow [our guide](https://unsloth.ai/docs/get-started/install). This notebook is licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).

You will learn how to do [data prep](#Data), how to [train](#Train), how to [run the model](#Inference), & how to save it

### News

Introducing **Unsloth Studio** - a new open source, no-code web UI to train and run LLMs. [Blog](https://unsloth.ai/docs/new/studio) • [Notebook](https://colab.research.google.com/github/unslothai/unsloth/blob/main/studio/Unsloth_Studio_Colab.ipynb)

<table><tr>
<td align="center"><a href="https://unsloth.ai/docs/new/studio"><img src="https://unsloth.ai/docs/~gitbook/image?url=https%3A%2F%2F3215535692-files.gitbook.io%2F~%2Ffiles%2Fv0%2Fb%2Fgitbook-x-prod.appspot.com%2Fo%2Fspaces%252FxhOjnexMCB3dmuQFQ2Zq%252Fuploads%252FxV1PO5DbF3ksB51nE2Tw%252Fmore%2520cropped%2520ui%2520for%2520homepage.png%3Falt%3Dmedia%26token%3Df75942c9-3d8d-4b59-8ba2-1a4a38de1b86&width=376&dpr=3&quality=100&sign=a663c397&sv=2" width="200" height="120" alt="Unsloth Studio Training UI"></a><br><sub><b>Train models</b> — no code needed</sub></td>
<td align="center"><a href="https://unsloth.ai/docs/new/studio"><img src="https://unsloth.ai/docs/~gitbook/image?url=https%3A%2F%2F3215535692-files.gitbook.io%2F~%2Ffiles%2Fv0%2Fb%2Fgitbook-x-prod.appspot.com%2Fo%2Fspaces%252FxhOjnexMCB3dmuQFQ2Zq%252Fuploads%252FRCnTAZ6Uh88DIlU3g0Ij%252Fmainpage%2520unsloth.png%3Falt%3Dmedia%26token%3D837c96b6-bd09-4e81-bc76-fa50421e9bfb&width=376&dpr=3&quality=100&sign=c1a39da1&sv=2" width="200" height="120" alt="Unsloth Studio Chat UI"></a><br><sub><b>Run GGUF models</b> on Mac, Windows & Linux</sub></td>
</tr></table>

Train MoEs - DeepSeek, GLM, Qwen and gpt-oss 12x faster with 35% less VRAM. [Blog](https://unsloth.ai/docs/new/faster-moe)

Ultra Long-Context Reinforcement Learning is here with 7x more context windows! [Blog](https://unsloth.ai/docs/new/grpo-long-context)

New in Reinforcement Learning: [FP8 RL](https://unsloth.ai/docs/new/fp8-reinforcement-learning) • [Vision RL](https://unsloth.ai/docs/new/vision-reinforcement-learning-vlm-rl) • [Standby](https://unsloth.ai/docs/basics/memory-efficient-rl) • [gpt-oss RL](https://unsloth.ai/docs/new/gpt-oss-reinforcement-learning)

Visit our docs for all our [model uploads](https://unsloth.ai/docs/get-started/unsloth-model-catalog) and [notebooks](https://unsloth.ai/docs/get-started/unsloth-notebooks).

### Installation

In [ ]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision bitsandbytes xformers==0.0.32.post2 \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
    !uv pip install -qqq --no-deps "torchcodec==0.7.0"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps "tokenizers>=0.22.0,<=0.23.0" trl==0.22.2 unsloth unsloth_zoo
!uv pip install transformers==5.2.0
# causal_conv1d is supported only on torch==2.8.0. If you have newer torch versions, please wait 10 minutes!
!uv pip install --no-build-isolation flash-linear-attention causal_conv1d==1.6.0
import torch
if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8:
    !uv pip install --no-deps "apache-tvm-ffi==0.1.9" "tilelang==0.1.8"
else:
    os.environ["FLA_TILELANG"] = "0"
!uv pip install --no-deps --upgrade "torchao>=0.16.0"

### Unsloth

In [ ]:
from unsloth import FastLanguageModel  # language-only SFT (was FastVisionModel)
import torch

# Putusan bodies are long — reconstructed inputs concatenate every verbatim span,
# so we train with a long context window. On an A100 80GB this is comfortable.
max_seq_length = 32768  # RoPE-scaled long context; raise if your p90 (measured below) needs it.

model, tokenizer = FastLanguageModel.from_pretrained(
    "unsloth/Qwen3.5-9B",              # Stage 1/2 base per RAG/ORCHESTRATION.md (was Qwen3.5-4B)
    max_seq_length = max_seq_length,
    load_in_4bit = False,             # 16-bit LoRA on A100 80GB (True = 4-bit QLoRA if VRAM-bound)
    load_in_8bit = False,
    full_finetuning = False,
    use_gradient_checkpointing = "unsloth",  # True or "unsloth" for long context
)

We now add LoRA adapters so we only train ~1% of parameters. This is a **language-only** extractor (the vision path is removed), so we target the attention + MLP projection modules directly.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 32,            # The larger, the higher the accuracy, but might overfit
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,   # Recommended alpha == r at least
    lora_dropout = 0,  # Supports any, but = 0 is optimized
    bias = "none",     # Supports any, but = "none" is optimized
    use_gradient_checkpointing = "unsloth",  # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,   # We support rank stabilized LoRA
    loftq_config = None,  # And LoftQ
)

<a name="Data"></a>
### Data Prep — Stage 0 (JSON-only)

We fine-tune a **structured section extractor**: given a putusan (court decision) body, emit one JSON object with exactly **31 verbatim-extractive sections**.

Per [`RAG/ORCHESTRATION.md`](../RAG/ORCHESTRATION.md) Stage 0, training reads **only** the extracted `LLM-aggregator/<CORPUS>/<MODEL>/output/*.json` — never the raw `.txt`. The model **input** is reconstructed from each record's own section spans, and the **target** is the 31-key `sections` JSON. `notebooks/build_dataset.py` does the dedup (by `source_sha256`) + 70/15/15 split (`sft` / `grpo` / frozen `benchmark`) and asserts every gold span is a substring of its reconstructed input.

We load the `sft.jsonl` split below (LaTeX-OCR dataset dropped entirely).

In [ ]:
# Stage 0: build data/train/{sft,grpo,benchmark}.jsonl from output/*.json if not present.
import os
from pathlib import Path

# Point this at the repo root (where LLM-aggregator/ and notebooks/ live).
REPO_ROOT = Path(os.environ.get("SINERGI_REPO_ROOT", "..")).resolve()
SFT_PATH  = REPO_ROOT / "data" / "train" / "sft.jsonl"

if not SFT_PATH.exists():
    !python "{REPO_ROOT / 'notebooks' / 'build_dataset.py'}" --repo-root "{REPO_ROOT}" --out-dir data/train

from datasets import load_dataset
dataset = load_dataset("json", data_files = str(SFT_PATH), split = "train")

Let's look at the dataset — each row is a chat with a `system` extraction instruction, the reconstructed putusan body as `user`, and the gold 31-section JSON as `assistant`.

In [ ]:
dataset

In [ ]:
# Peek at the assistant target (gold JSON) for the first example.
print(dataset[0]["messages"][2]["content"][:1000])

We format each chat into a single `text` string with the model's chat template, so the standard text `SFTTrainer` can tokenize it (no vision collator needed).

In [ ]:
def formatting_prompts_func(examples):
    texts = [
        tokenizer.apply_chat_template(msgs, tokenize = False, add_generation_prompt = False)
        for msgs in examples["messages"]
    ]
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched = True)

Measure the tokenized length distribution and set `max_length` from the **90th percentile** (per ORCHESTRATION Stage 1), capped at the model's `max_seq_length`.

In [ ]:
import numpy as np

token_lengths = [len(tokenizer(t, add_special_tokens = False)["input_ids"]) for t in dataset["text"]]
p50, p90, p95 = (int(np.percentile(token_lengths, p)) for p in (50, 90, 95))
# Round p90 up to a multiple of 256; never exceed the model's context window.
MAX_LENGTH = int(min(np.ceil(p90 / 256) * 256, max_seq_length))
print(f"token length  p50={p50}  p90={p90}  p95={p95}  max={max(token_lengths)}")
print(f"MAX_LENGTH (90th pct, capped at {max_seq_length}) = {MAX_LENGTH}")

Here is the fully-formatted `text` for the first example (system + user putusan body + assistant JSON):

In [ ]:
print(dataset[0]["text"][:2000])

Before finetuning, let's see what the base model emits for the first putusan body (system + user only, assistant left blank).

In [ ]:
FastLanguageModel.for_inference(model)  # Enable for inference!

# system + user (drop the gold assistant turn); ask the model to produce the JSON.
prompt_messages = dataset[0]["messages"][:2]
inputs = tokenizer.apply_chat_template(
    prompt_messages,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(input_ids = inputs, streamer = text_streamer, max_new_tokens = 512,
                   use_cache = True, temperature = 0.7, min_p = 0.1)

<a name="Train"></a>
### Train the model

Standard text `SFTTrainer` (no `UnslothVisionDataCollator`). We train on the `text` field for `num_train_epochs = 2` (ORCHESTRATION Stage 1 says 1-3), with `max_length` set to the measured 90th-percentile token length. We then use `train_on_responses_only` so the loss is computed **only on the assistant JSON**, not on the (very long) putusan input. Set `max_steps` for a quick smoke test.

In [ ]:
from trl import SFTTrainer, SFTConfig

FastLanguageModel.for_training(model)  # Enable for training!

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 1,   # long sequences - keep the micro-batch small
        gradient_accumulation_steps = 8,   # effective batch size = 8
        warmup_steps = 5,
        num_train_epochs = 2,              # 1-3 per ORCHESTRATION Stage 1
        # max_steps = 30,                  # uncomment for a quick smoke test
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "wandb",     # For Weights and Biases
        max_length = MAX_LENGTH,
    ),
)

In [ ]:
# Train only on the assistant response (the JSON), masking the putusan input.
# Qwen3.5 uses ChatML markers.
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part    = "<|im_start|>assistant\n",
)

In [ ]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

In [ ]:
trainer_stats = trainer.train()

In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

<a name="Inference"></a>
### Inference
Run the finetuned extractor on a held-out putusan body - it should emit the 31-section JSON. We use low-temperature decoding (`temperature = 0.3`, `min_p = 0.1`) for stable structured output.

In [ ]:
FastLanguageModel.for_inference(model)  # Enable for inference!

# Try a benchmark-split example if available, else reuse a training example.
_bench = REPO_ROOT / "data" / "train" / "benchmark.jsonl"
if _bench.exists():
    import json as _json
    with _bench.open(encoding="utf-8") as f:
        _ex = _json.loads(f.readline())
    prompt_messages = _ex["messages"][:2]
else:
    prompt_messages = dataset[0]["messages"][:2]

inputs = tokenizer.apply_chat_template(
    prompt_messages,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(input_ids = inputs, streamer = text_streamer, max_new_tokens = 2048,
                   use_cache = True, temperature = 0.3, min_p = 0.1)

<a name="Save"></a>
### Saving, loading finetuned models
Save the Stage-1 LoRA adapters as `qwen_extractor_sft_lora` - Stage 2 (GRPO) continues from this. Use `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, not the full model. To save 16-bit merged for serving, scroll down.

In [ ]:
model.save_pretrained("qwen_extractor_sft_lora")  # Local saving (Stage 2 continues from this)
tokenizer.save_pretrained("qwen_extractor_sft_lora")
# model.push_to_hub("your_name/qwen_extractor_sft_lora", token = "YOUR_HF_TOKEN") # Online saving
# tokenizer.push_to_hub("your_name/qwen_extractor_sft_lora", token = "YOUR_HF_TOKEN") # Online saving

Now if you want to load the LoRA adapters we just saved for inference, set `False` to `True`:

In [ ]:
if False:
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "qwen_extractor_sft_lora", # YOUR MODEL YOU USED FOR TRAINING
        max_seq_length = max_seq_length,
        load_in_4bit = False, # Set to False for 16bit LoRA
    )
    FastLanguageModel.for_inference(model) # Enable for inference!

    prompt_messages = dataset[0]["messages"][:2]
    inputs = tokenizer.apply_chat_template(
        prompt_messages, add_generation_prompt = True, return_tensors = "pt",
    ).to("cuda")
    from transformers import TextStreamer
    text_streamer = TextStreamer(tokenizer, skip_prompt = True)
    _ = model.generate(input_ids = inputs, streamer = text_streamer, max_new_tokens = 2048,
                       use_cache = True, temperature = 0.3, min_p = 0.1)

### Saving to float16 for vLLM

We also support saving to `float16` directly for serving (Stage 3). Select `merged_16bit` for float16. Use `push_to_hub_merged` to upload to your Hugging Face account. See [our docs](https://unsloth.ai/docs/basics/inference-and-deployment) for more deployment options.

In [ ]:
# Select ONLY 1 to save! (Both not needed!)

# Save locally to 16bit merged (serving-ready extractor)
if False: model.save_pretrained_merged("qwen_extractor_sft_merged", tokenizer,)

# To export and save to your Hugging Face account
if False: model.push_to_hub_merged("YOUR_USERNAME/qwen_extractor_sft_merged", tokenizer, token = "YOUR_HF_TOKEN")